# Day 19 — Prompt Engineering for Reliable LLM Outputs

## Objective

Systematically evaluate five prompt engineering techniques
on the same question-answering task.

### Techniques Tested

1. Role Assignment
2. Output Format Specification
3. Reasoning Instruction
4. Few-Shot Examples
5. Negative Constraints

### Evaluation Metrics

- Accuracy: 1–5
- Format Consistency: 1–5

The experiment also evaluates a grounding prompt against
five out-of-context questions.

In [1]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Check your .env file."
    )

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.5"

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [2]:
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI

print("Python:", sys.executable)

load_dotenv(
    dotenv_path=".env",
    override=True
)

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", bool(api_key))
print("API key prefix:", api_key[:7] if api_key else None)

Python: c:\Users\ansh\AppData\Local\Python\pythoncore-3.14-64\python.exe
API key loaded: True
API key prefix: sk-proj


## 1. Task Definition

### Task: Company Policy Question Answering

The model must answer questions using only the provided
NovaTech company policy context.

This task was selected because it directly represents
a common RAG use case where accuracy, consistency,
and grounding are important.

In [3]:
client = OpenAI(api_key=api_key)

MODEL = "gpt-5.5"

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [6]:
# API test
# Run this only after API credits are available.

print("API test is currently skipped because the API account has no credits.")

API test is currently skipped because the API account has no credits.


# Day 19 — Prompt Engineering for Reliable LLM Outputs

## Task Definition

### Task
Company Policy Question Answering

The objective is to determine how different prompt engineering
techniques affect the accuracy and format consistency of an LLM
when answering questions from a fixed company policy context.

### Techniques Tested

1. Baseline
2. Role Assignment
3. Output Format Specification
4. Reasoning Instruction
5. Few-Shot Examples
6. Negative Constraints

### Evaluation Metrics

Each response will be evaluated on:

- Accuracy: 1–5
- Format Consistency: 1–5

Average Score = (Accuracy + Format Consistency) / 2

In [7]:
knowledge_base = {
    "vacation_policy": """
NovaTech employees receive 20 paid vacation days per year.
Unused vacation days can be carried forward up to a maximum
of 10 days into the next year.
Employees must request vacation at least 5 business days
in advance.
""",

    "remote_work": """
NovaTech employees may work remotely up to 3 days per week.
Employees must work from the office on Mondays and Wednesdays.
Remote work requires manager approval.
""",

    "parental_leave": """
NovaTech provides 16 weeks of paid parental leave to eligible
full-time employees.
""",

    "expense_policy": """
Business expenses must be submitted within 30 days.
Receipts are required for expenses above $25.
Manager approval is required for expenses above $500.
""",

    "security_policy": """
Employees must use multi-factor authentication for company
systems.
Passwords must be at least 12 characters long.
Employees must not share passwords with other people.
"""
}

context = "\n\n".join(knowledge_base.values())

print(context)


NovaTech employees receive 20 paid vacation days per year.
Unused vacation days can be carried forward up to a maximum
of 10 days into the next year.
Employees must request vacation at least 5 business days
in advance.



NovaTech employees may work remotely up to 3 days per week.
Employees must work from the office on Mondays and Wednesdays.
Remote work requires manager approval.



NovaTech provides 16 weeks of paid parental leave to eligible
full-time employees.



Business expenses must be submitted within 30 days.
Receipts are required for expenses above $25.
Manager approval is required for expenses above $500.



Employees must use multi-factor authentication for company
systems.
Passwords must be at least 12 characters long.
Employees must not share passwords with other people.



In [8]:
test_questions = [
    "How many paid vacation days do NovaTech employees receive?",
    "How many vacation days can be carried forward?",
    "How many days in advance must vacation be requested?",
    "How many days per week can employees work remotely?",
    "Which days require employees to work from the office?",
    "How many weeks of paid parental leave are provided?",
    "When must business expenses be submitted?",
    "When are receipts required for expenses?",
    "When is manager approval required for expenses?",
    "What authentication method must employees use for company systems?"
]

print(f"Total test questions: {len(test_questions)}")

Total test questions: 10


In [9]:
expected_answers = [
    "20 paid vacation days per year.",
    "Up to 10 unused vacation days can be carried forward.",
    "Vacation must be requested at least 5 business days in advance.",
    "Employees can work remotely up to 3 days per week.",
    "Employees must work from the office on Mondays and Wednesdays.",
    "Eligible full-time employees receive 16 weeks of paid parental leave.",
    "Business expenses must be submitted within 30 days.",
    "Receipts are required for expenses above $25.",
    "Manager approval is required for expenses above $500.",
    "Employees must use multi-factor authentication."
]

print(f"Total expected answers: {len(expected_answers)}")

Total expected answers: 10


## Evaluation Rubric

### Accuracy

| Score | Meaning |
|------:|---------|
| 1 | Completely incorrect |
| 2 | Mostly incorrect |
| 3 | Partially correct |
| 4 | Mostly correct |
| 5 | Fully correct |

### Format Consistency

| Score | Meaning |
|------:|---------|
| 1 | Highly inconsistent |
| 2 | Poor consistency |
| 3 | Acceptable |
| 4 | Consistent |
| 5 | Highly consistent |

### Overall Score

Average Score = (Accuracy + Format Consistency) / 2

In [10]:
prompts = {
    "baseline": """
Answer the user's question using the provided context.

Context:
{context}

Question:
{question}

Answer:
""",

    "role_assignment": """
You are an expert company policy assistant.

Answer the user's question accurately using only the provided context.

Context:
{context}

Question:
{question}

Answer:
""",

    "output_format": """
Answer the user's question using the provided context.

Return exactly:

Answer: <direct answer>

Do not add unnecessary explanation.

Context:
{context}

Question:
{question}
""",

    "reasoning_instruction": """
Answer the user's question using the provided context.

Before answering, carefully verify which facts in the context
directly support the answer. Do not reveal internal reasoning.

Context:
{context}

Question:
{question}

Answer:
""",

    "few_shot": """
Answer company policy questions using the provided context.

Example:

Context:
Employees receive 20 paid vacation days per year.

Question:
How many vacation days do employees receive?

Answer:
20 paid vacation days per year.

Now answer the following question.

Context:
{context}

Question:
{question}

Answer:
""",

    "negative_constraints": """
Answer the user's question using only the provided context.

Rules:
- Do not use outside knowledge.
- Do not invent facts.
- Do not guess missing information.
- Do not add unsupported details.
- Keep the answer concise.
- If the context does not contain the answer, say:
  "The provided context does not contain enough information."

Context:
{context}

Question:
{question}

Answer:
"""
}

print("Prompt versions:", list(prompts.keys()))

Prompt versions: ['baseline', 'role_assignment', 'output_format', 'reasoning_instruction', 'few_shot', 'negative_constraints']


In [11]:
def generate_answer(prompt_template, question):
    prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = client.responses.create(
        model=MODEL,
        input=prompt
    )

    return response.output_text.strip()

In [12]:
def run_prompt_experiment(prompt_name):
    results = []

    prompt_template = prompts[prompt_name]

    for question in test_questions:
        answer = generate_answer(
            prompt_template,
            question
        )

        results.append({
            "prompt_version": prompt_name,
            "question": question,
            "answer": answer
        })

    return pd.DataFrame(results)

In [13]:
def run_prompt_experiment(prompt_name):
    results = []

    prompt_template = prompts[prompt_name]

    for question in test_questions:
        answer = generate_answer(
            prompt_template,
            question
        )

        results.append({
            "prompt_version": prompt_name,
            "question": question,
            "answer": answer
        })

    return pd.DataFrame(results)

In [14]:
def evaluate_answer(question, expected, answer):
    evaluation_prompt = f"""
Evaluate the model answer against the expected answer.

Question:
{question}

Expected Answer:
{expected}

Model Answer:
{answer}

Score the response:

Accuracy: 1-5
Format Consistency: 1-5

Return ONLY JSON:

{{
    "accuracy": <integer>,
    "format_consistency": <integer>
}}
"""

    response = client.responses.create(
        model=MODEL,
        input=evaluation_prompt
    )

    return json.loads(response.output_text)

In [15]:
experiment_results = {}

for prompt_name in prompts:
    experiment_results[prompt_name] = {
        "description": "",
        "accuracy_scores": [],
        "format_scores": [],
        "average_score": None
    }

print(experiment_results.keys())

dict_keys(['baseline', 'role_assignment', 'output_format', 'reasoning_instruction', 'few_shot', 'negative_constraints'])


In [16]:
prompt_versions = {
    "baseline": {
        "average_score": None,
        "description": "Simple question-answering prompt."
    },

    "role_assignment": {
        "average_score": None,
        "description": "Added an expert company-policy assistant role."
    },

    "output_format": {
        "average_score": None,
        "description": "Added an explicit response format."
    },

    "reasoning_instruction": {
        "average_score": None,
        "description": "Added internal fact verification before answering."
    },

    "few_shot": {
        "average_score": None,
        "description": "Added an example demonstrating the desired answer."
    },

    "negative_constraints": {
        "average_score": None,
        "description": "Added constraints against guessing and unsupported information."
    }
}

prompt_versions

{'baseline': {'average_score': None,
  'description': 'Simple question-answering prompt.'},
 'role_assignment': {'average_score': None,
  'description': 'Added an expert company-policy assistant role.'},
 'output_format': {'average_score': None,
  'description': 'Added an explicit response format.'},
 'reasoning_instruction': {'average_score': None,
  'description': 'Added internal fact verification before answering.'},
 'few_shot': {'average_score': None,
  'description': 'Added an example demonstrating the desired answer.'},
 'negative_constraints': {'average_score': None,
  'description': 'Added constraints against guessing and unsupported information.'}}

# Grounding System Prompt

The following system prompt is designed for a RAG assistant.
Its purpose is to prevent the assistant from answering questions
using unsupported information from its pretrained knowledge.

In [17]:
grounding_system_prompt = """
You are a grounded question-answering assistant.

Your only source of truth is the context provided to you.

Rules:

1. Answer only when the answer is directly supported by the context.
2. Do not use outside knowledge.
3. Do not use information from training data to fill missing facts.
4. Do not guess.
5. Do not infer unsupported information.
6. If the answer is not supported by the context, refuse to answer.

When the context does not contain the answer, respond:

"I cannot answer this question because the provided context
does not contain the required information."

For supported questions, answer concisely and factually.
"""

print(grounding_system_prompt)


You are a grounded question-answering assistant.

Your only source of truth is the context provided to you.

Rules:

1. Answer only when the answer is directly supported by the context.
2. Do not use outside knowledge.
3. Do not use information from training data to fill missing facts.
4. Do not guess.
5. Do not infer unsupported information.
6. If the answer is not supported by the context, refuse to answer.

When the context does not contain the answer, respond:

"I cannot answer this question because the provided context
does not contain the required information."

For supported questions, answer concisely and factually.



In [18]:
out_of_context_questions = [
    "Who is the CEO of NovaTech?",
    "What is NovaTech's annual revenue?",
    "When was NovaTech founded?",
    "What programming language does NovaTech use internally?",
    "How many offices does NovaTech have worldwide?"
]

print(f"Out-of-context tests: {len(out_of_context_questions)}")

Out-of-context tests: 5


In [19]:
def grounded_answer(question):
    prompt = f"""
{grounding_system_prompt}

Context:
{context}

Question:
{question}
"""

    response = client.responses.create(
        model=MODEL,
        input=prompt
    )

    return response.output_text.strip()

In [22]:
print("Grounding section ready.")

Grounding section ready.


In [23]:
print("Grounding section ready.")

Grounding section ready.


In [25]:
import pandas as pd

print("Pandas imported successfully.")

Pandas imported successfully.


In [26]:
grounding_results = []

for question in out_of_context_questions:
    grounding_results.append({
        "question": question,
        "answer": "API credits required for actual test",
        "refused_correctly": None
    })

grounding_df = pd.DataFrame(grounding_results)

display(grounding_df)

,question,answer,refused_correctly
0,Who is the CEO of NovaTech?,API credits required for actual test,None
1,What is NovaTech's annual revenue?,API credits required for actual test,None
2,When was NovaTech founded?,API credits required for actual test,None
3,What programming language does NovaTech use in...,API credits required for actual test,None
4,How many offices does NovaTech have worldwide?,API credits required for actual test,None


In [27]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.5"

In [28]:
grounding_df

,question,answer,refused_correctly
0,Who is the CEO of NovaTech?,API credits required for actual test,None
1,What is NovaTech's annual revenue?,API credits required for actual test,None
2,When was NovaTech founded?,API credits required for actual test,None
3,What programming language does NovaTech use in...,API credits required for actual test,None
4,How many offices does NovaTech have worldwide?,API credits required for actual test,None


In [29]:
correct_refusals = grounding_df["refused_correctly"].notna().sum()
total_tests = len(grounding_df)

print(f"Grounding tests prepared: {total_tests}")
print("Actual API evaluation will be performed after credits are available.")

Grounding tests prepared: 5
Actual API evaluation will be performed after credits are available.


In [30]:
evaluation_results = {
    "baseline": [],
    "role_assignment": [],
    "output_format": [],
    "reasoning_instruction": [],
    "few_shot": [],
    "negative_constraints": []
}

print("Evaluation structure created.")

Evaluation structure created.


In [31]:
def calculate_average_score(accuracy_scores, format_scores):
    if not accuracy_scores or not format_scores:
        return None

    accuracy_avg = sum(accuracy_scores) / len(accuracy_scores)
    format_avg = sum(format_scores) / len(format_scores)

    return round((accuracy_avg + format_avg) / 2, 2)

In [32]:
test_score = calculate_average_score(
    [5, 4, 5],
    [5, 4, 5]
)

print("Test average score:", test_score)

Test average score: 4.67


In [33]:
prompt_versions = {
    "baseline": {
        "average_score": None,
        "description": "Simple question-answering prompt."
    },

    "role_assignment": {
        "average_score": None,
        "description": "Added an expert company-policy assistant role."
    },

    "output_format": {
        "average_score": None,
        "description": "Specified an exact response format."
    },

    "reasoning_instruction": {
        "average_score": None,
        "description": "Added internal fact verification before answering."
    },

    "few_shot": {
        "average_score": None,
        "description": "Added an example demonstrating the desired answer."
    },

    "negative_constraints": {
        "average_score": None,
        "description": "Added constraints against guessing and unsupported information."
    }
}

display(prompt_versions)

{'baseline': {'average_score': None,
  'description': 'Simple question-answering prompt.'},
 'role_assignment': {'average_score': None,
  'description': 'Added an expert company-policy assistant role.'},
 'output_format': {'average_score': None,
  'description': 'Specified an exact response format.'},
 'reasoning_instruction': {'average_score': None,
  'description': 'Added internal fact verification before answering.'},
 'few_shot': {'average_score': None,
  'description': 'Added an example demonstrating the desired answer.'},
 'negative_constraints': {'average_score': None,
  'description': 'Added constraints against guessing and unsupported information.'}}

In [34]:
comparison_df = pd.DataFrame({
    "Technique": [
        "Baseline",
        "Role Assignment",
        "Output Format",
        "Reasoning Instruction",
        "Few-Shot Examples",
        "Negative Constraints"
    ],
    
    "Accuracy": [
        None,
        None,
        None,
        None,
        None,
        None
    ],
    
    "Format Consistency": [
        None,
        None,
        None,
        None,
        None,
        None
    ],
    
    "Average Score": [
        None,
        None,
        None,
        None,
        None,
        None
    ]
})

display(comparison_df)

,Technique,Accuracy,Format Consistency,Average Score
0,Baseline,None,None,None
1,Role Assignment,None,None,None
2,Output Format,None,None,None
3,Reasoning Instruction,None,None,None
4,Few-Shot Examples,None,None,None
5,Negative Constraints,None,None,None


In [35]:
def calculate_improvement(baseline_score, technique_score):
    if baseline_score is None or technique_score is None:
        return None

    return round(technique_score - baseline_score, 2)

In [36]:
def find_best_technique(comparison):
    valid_results = comparison.dropna(
        subset=["Average Score"]
    )

    if valid_results.empty:
        return None

    best_row = valid_results.loc[
        valid_results["Average Score"].idxmax()
    ]

    return {
        "technique": best_row["Technique"],
        "score": best_row["Average Score"]
    }

In [37]:
best = find_best_technique(comparison_df)

print("Best technique:", best)

Best technique: None


In [38]:
def run_full_experiment():

    all_results = {}

    for prompt_name, prompt_template in prompts.items():

        print(f"Running: {prompt_name}")

        prompt_results = []

        for question in test_questions:

            answer = generate_answer(
                prompt_template,
                question
            )

            prompt_results.append({
                "question": question,
                "answer": answer
            })

        all_results[prompt_name] = prompt_results

    return all_results

In [39]:
def score_all_results(all_results):

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

In [40]:
def score_all_results(all_results):

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

In [41]:
def build_comparison(scored_results):

    rows = []

    for prompt_name, results in scored_results.items():

        accuracy = sum(
            r["accuracy"] for r in results
        ) / len(results)

        format_score = sum(
            r["format_consistency"] for r in results
        ) / len(results)

        average = (accuracy + format_score) / 2

        rows.append({
            "Technique": prompt_name,
            "Accuracy": round(accuracy, 2),
            "Format Consistency": round(format_score, 2),
            "Average Score": round(average, 2)
        })

    return pd.DataFrame(rows)

In [45]:
API_AVAILABLE = False

def run_full_experiment():

    if not API_AVAILABLE:
        print("API experiment skipped.")
        print("Reason: OpenAI API credits are unavailable.")
        return {}

    all_results = {}

    for prompt_name, prompt_template in prompts.items():

        print(f"Running: {prompt_name}")

        prompt_results = []

        for question in test_questions:

            answer = generate_answer(
                prompt_template,
                question
            )

            prompt_results.append({
                "question": question,
                "answer": answer
            })

        all_results[prompt_name] = prompt_results

    return all_results

print("Safe experiment runner loaded.")

Safe experiment runner loaded.


In [46]:
def score_all_results(all_results):

    if not all_results:
        print("No API results available for scoring.")
        return {}

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

print("Safe scoring function loaded.")

Safe scoring function loaded.


In [47]:
all_results = run_full_experiment()

print("Result:", all_results)

API experiment skipped.
Reason: OpenAI API credits are unavailable.
Result: {}


In [48]:
API_AVAILABLE = False

def run_full_experiment():

    if not API_AVAILABLE:
        print("API experiment skipped.")
        print("Reason: OpenAI API credits are unavailable.")
        return {}

    all_results = {}

    for prompt_name, prompt_template in prompts.items():

        print(f"Running: {prompt_name}")

        prompt_results = []

        for question in test_questions:

            answer = generate_answer(
                prompt_template,
                question
            )

            prompt_results.append({
                "question": question,
                "answer": answer
            })

        all_results[prompt_name] = prompt_results

    return all_results

print("Safe experiment runner loaded.")

Safe experiment runner loaded.


In [49]:
def score_all_results(all_results):

    if not all_results:
        print("No API results available for scoring.")
        return {}

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

print("Safe scoring function loaded.")

Safe scoring function loaded.


In [50]:
def score_all_results(all_results):

    if not all_results:
        print("No API results available for scoring.")
        return {}

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

print("Safe scoring function loaded.")

Safe scoring function loaded.


In [51]:
def score_all_results(all_results):

    if not all_results:
        print("No API results available for scoring.")
        return {}

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

print("Safe scoring function loaded.")

Safe scoring function loaded.


In [52]:
scored_results = score_all_results(all_results)

No API results available for scoring.


In [53]:
import sys
print(sys.executable)

c:\Users\ansh\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [54]:
print("Safe experiment runner loaded.")

Safe experiment runner loaded.


In [55]:
all_results = run_full_experiment()

API experiment skipped.
Reason: OpenAI API credits are unavailable.


In [56]:
API_AVAILABLE = False

def run_full_experiment():

    if not API_AVAILABLE:
        print("API experiment skipped.")
        print("Reason: OpenAI API credits are unavailable.")
        return {}

    all_results = {}

    for prompt_name, prompt_template in prompts.items():

        print(f"Running: {prompt_name}")

        prompt_results = []

        for question in test_questions:

            answer = generate_answer(
                prompt_template,
                question
            )

            prompt_results.append({
                "question": question,
                "answer": answer
            })

        all_results[prompt_name] = prompt_results

    return all_results

print("Safe experiment runner loaded.")

Safe experiment runner loaded.


In [57]:
all_results = run_full_experiment()
print(all_results)

API experiment skipped.
Reason: OpenAI API credits are unavailable.
{}


In [58]:
def score_all_results(all_results):

    if not all_results:
        print("No API results available for scoring.")
        return {}

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

print("Safe scoring function loaded.")

Safe scoring function loaded.


In [59]:
def score_all_results(all_results):

    if not all_results:
        print("No API results available for scoring.")
        return {}

    scored_results = {}

    for prompt_name, results in all_results.items():

        scores = []

        for index, result in enumerate(results):

            evaluation = evaluate_answer(
                test_questions[index],
                expected_answers[index],
                result["answer"]
            )

            scores.append({
                "question": result["question"],
                "answer": result["answer"],
                "accuracy": evaluation["accuracy"],
                "format_consistency": evaluation["format_consistency"]
            })

        scored_results[prompt_name] = scores

    return scored_results

print("Safe scoring function loaded.")

Safe scoring function loaded.


In [60]:
scored_results = score_all_results(all_results)
print(scored_results)

No API results available for scoring.
{}


In [61]:
scored_results = score_all_results(all_results)
print(scored_results)

No API results available for scoring.
{}


In [62]:
import sys

print(sys.executable)

c:\Users\ansh\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [63]:
def build_comparison_table(scored_results):

    rows = []

    technique_names = {
        "baseline": "Baseline",
        "role_assignment": "Role Assignment",
        "output_format": "Output Format",
        "reasoning_instruction": "Reasoning Instruction",
        "few_shot": "Few-Shot Examples",
        "negative_constraints": "Negative Constraints"
    }

    for prompt_name in prompts.keys():

        if prompt_name not in scored_results:
            rows.append({
                "Technique": technique_names[prompt_name],
                "Accuracy": None,
                "Format Consistency": None,
                "Average Score": None
            })
            continue

        results = scored_results[prompt_name]

        accuracy_scores = [
            item["accuracy"]
            for item in results
        ]

        format_scores = [
            item["format_consistency"]
            for item in results
        ]

        average_score = calculate_average_score(
            accuracy_scores,
            format_scores
        )

        rows.append({
            "Technique": technique_names[prompt_name],
            "Accuracy": round(sum(accuracy_scores) / len(accuracy_scores), 2),
            "Format Consistency": round(sum(format_scores) / len(format_scores), 2),
            "Average Score": average_score
        })

    return pd.DataFrame(rows)

In [64]:
comparison_df = build_comparison_table(scored_results)

display(comparison_df)

,Technique,Accuracy,Format Consistency,Average Score
0,Baseline,None,None,None
1,Role Assignment,None,None,None
2,Output Format,None,None,None
3,Reasoning Instruction,None,None,None
4,Few-Shot Examples,None,None,None
5,Negative Constraints,None,None,None


In [65]:
best_technique = find_best_technique(comparison_df)

print("Best-performing technique:")
print(best_technique)

Best-performing technique:
None


In [66]:
baseline_score = comparison_df.loc[
    comparison_df["Technique"] == "Baseline",
    "Average Score"
].iloc[0]

comparison_df["Improvement vs Baseline"] = comparison_df[
    "Average Score"
].apply(
    lambda score: calculate_improvement(baseline_score, score)
)

display(comparison_df)

,Technique,Accuracy,Format Consistency,Average Score,Improvement vs Baseline
0,Baseline,None,None,None,None
1,Role Assignment,None,None,None,None
2,Output Format,None,None,None,None
3,Reasoning Instruction,None,None,None,None
4,Few-Shot Examples,None,None,None,None
5,Negative Constraints,None,None,None,None


## Analysis and Conclusion

The experiment was designed to compare six prompt versions on the same
10-question company-policy question-answering dataset.

The tested techniques were:

1. Baseline prompting
2. Role assignment
3. Output format specification
4. Reasoning instruction
5. Few-shot examples
6. Negative constraints

Each version was intended to improve either answer accuracy, output
consistency, or resistance to unsupported information.

### Largest Improvement

The largest improvement will be identified after running the experiment
with the OpenAI API and comparing each technique against the baseline.

### Hypothesis

Negative constraints and few-shot examples are expected to be particularly
useful because they explicitly guide the model toward the desired behavior
and reduce guessing or unsupported responses.

### Grounding Experiment

A separate grounding prompt was designed for a RAG assistant. The prompt
instructs the model to use only the supplied context and refuse questions
whose answers are not directly supported by that context.

Five out-of-context questions were prepared to test whether the assistant
would avoid relying on its pretrained knowledge.

### API Limitation

The actual OpenAI API evaluation could not be executed in this run because
the API organization had exhausted its available credit balance.

Therefore, no fabricated scores or model responses are reported.
The notebook contains the complete evaluation pipeline and can be rerun
once API credits are available.

In [67]:
import json

results_summary = {
    "day": 19,
    "title": "Prompt Engineering for Reliable LLM Outputs",
    "task": "Company Policy Question Answering",
    "total_test_questions": len(test_questions),
    "prompt_versions": list(prompts.keys()),
    "api_experiment_executed": bool(all_results),
    "api_available": API_AVAILABLE,
    "comparison": comparison_df.to_dict(orient="records"),
    "grounding_questions": out_of_context_questions,
    "grounding_results": grounding_results
}

with open("day_19_results.json", "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)

print("day_19_results.json created successfully.")

day_19_results.json created successfully.


# Day 19 — Prompt Engineering for Reliable LLM Outputs

## Objective

Systematically evaluate different prompt engineering techniques on the
same company-policy question-answering task and measure their effect on
accuracy and format consistency.

## Task

The experiment uses a fictional NovaTech company-policy knowledge base.

The model answers 10 questions using the supplied context.

## Prompt Techniques Tested

1. Baseline
2. Role Assignment
3. Output Format Specification
4. Reasoning Instruction
5. Few-Shot Examples
6. Negative Constraints

## Evaluation

Each answer is evaluated using two metrics:

- Accuracy: 1–5
- Format Consistency: 1–5

Overall score:

```text
Average Score = (Accuracy + Format Consistency) / 2